# 🚀 Boosting Methods — Concepts & Theory
**Topic:** Ensemble Learning | AdaBoost | Gradient Boosting

---

## 📌 Is Notebook Mein Kya Sikhenge?

| # | Topic | Kya Hai |
|---|-------|---------|
| 1 | Introduction to Boosting | Weak learners ko strong kaise banate hain |
| 2 | Boosting Methods Overview | AdaBoost, GBM, XGBoost ka comparison |
| 3 | AdaBoost Theory | Sample weights, alpha calculation |
| 4 | AdaBoost Implementation | sklearn se complete code |
| 5 | Gradient Boosting Theory | Residuals, pseudo-residuals |
| 6 | Gradient Boosting Implementation | sklearn se complete code |

---

### 💡 Ek Simple Analogy Se Shuru Karte Hain

**Exam Ki Tayyari:**
- Pehle teacher ne poora syllabus padhaya → kuch samajh aaya, kuch nahi
- Doosre teacher ne sirf jo miss hua woh padhaya → thoda aur clear hua
- Teesre teacher ne remaining gaps fill kiye → ab sab clear!

Yahi **Boosting** hai! Har naya model apne **predecessors ki galtiyon** pe focus karta hai. 🎯

---
## 1️⃣ Introduction to Boosting

### 🔑 Naye Keywords — Pehle Samjho!

| Term | Matlab (Simple) |
|------|----------------|
| **Ensemble** | Multiple models ko combine karna — ek team ki tarah |
| **Weak Learner** | Ek model jo barely better than random guessing karta hai (e.g., decision stump) |
| **Strong Learner** | High accuracy wala final model |
| **Boosting** | Weak learners ko sequentially combine karna — har model pichle ki galti fix karta hai |
| **Bagging** | Models ko parallel train karna (e.g., Random Forest) |
| **Decision Stump** | Ek decision tree with only ONE split (depth=1) — sabse common weak learner |

### Boosting vs Bagging

| Feature | Bagging (Random Forest) | Boosting (AdaBoost, GBM) |
|---------|------------------------|-------------------------|
| Training | Parallel | Sequential |
| Each model | Independent | Depends on previous model |
| Focus | Variance reduction | Bias reduction |
| Overfitting risk | Low | Medium (needs tuning) |
| Speed | Faster | Slower |
| Accuracy | Good | Usually better |

### How Boosting Works — Step by Step

```
Step 1: Weak Model 1 fit karo sare data pe
          → Kuch samples galat predict hua

Step 2: Galat samples ko ZYADA weight do
          → Weak Model 2 inpe focus karta hai

Step 3: Phir galat samples ko aur zyada weight
          → Weak Model 3 unpe focus karta hai

Step 4: Sab models ko weighted vote karne do
          → Final prediction!
```

---
### 🎯 Yeh Humne Kyun Seekha?
> Real world mein single model akela kaafi nahi hota. Boosting multiple weak models ko ek powerful model mein combine karta hai. Yeh tabular data (CSV files) pe best perform karne wala approach hai — isliye industry mein most popular hai.

**Real Life Example:** Credit card fraud detection. Ek model 70% accurate, dusra 65%, teesra 68% — lekin sab ko smart tarike se combine karo → 95%+ accuracy!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ==============================================
# Sample Dataset: Customer Loan Default
# ==============================================
np.random.seed(42)

data = {
    'Age':         [23, 45, 31, 56, 28, 42, 35, 60, 27, 48, 33, 52, 29, 41, 38],
    'Income_L':    [2.5, 8.0, 4.5, 12.0, 3.0, 7.5, 5.0, 15.0, 2.8, 9.0, 4.0, 11.0, 3.2, 6.5, 5.5],
    'Debt_Ratio':  [0.8, 0.2, 0.5, 0.1, 0.9, 0.3, 0.4, 0.1, 0.85, 0.25, 0.6, 0.15, 0.75, 0.35, 0.45],
    'Credit_Score':[550, 780, 650, 820, 520, 760, 680, 850, 510, 790, 620, 810, 540, 740, 690],
    'Default':     [1,   0,   1,   0,   1,   0,   0,   0,   1,   0,   1,   0,   1,   0,   0]
}
df = pd.DataFrame(data)

print("=== Sample Loan Dataset ===")
print(df.to_string())
print(f"\nDefault: {df.Default.sum()} | No Default: {(df.Default==0).sum()}")

In [ ]:
# Bagging vs Boosting ka visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bagging
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 6)
axes[0].set_title('Bagging (Parallel — e.g. Random Forest)', fontsize=11, fontweight='bold')
axes[0].axis('off')
# Data box
axes[0].add_patch(plt.Rectangle((3.5, 4.5), 3, 1, fill=True, color='#3498db', alpha=0.7))
axes[0].text(5, 5, 'Training Data', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
# 3 parallel models
for i, (x, label) in enumerate([(1, 'Model 1'), (4.5, 'Model 2'), (8, 'Model 3')]):
    axes[0].add_patch(plt.Rectangle((x-0.8, 2.5), 1.6, 1, fill=True, color='#2ecc71', alpha=0.7))
    axes[0].text(x, 3, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    axes[0].annotate('', xy=(x, 3.5), xytext=(5, 4.5), arrowprops=dict(arrowstyle='->', color='gray'))
# Final prediction
axes[0].add_patch(plt.Rectangle((3.5, 0.5), 3, 1, fill=True, color='#e74c3c', alpha=0.7))
axes[0].text(5, 1, 'Vote / Average', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
for x in [1, 4.5, 8]:
    axes[0].annotate('', xy=(5, 1.5), xytext=(x, 2.5), arrowprops=dict(arrowstyle='->', color='gray'))

# Boosting  
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 6)
axes[1].set_title('Boosting (Sequential — e.g. AdaBoost)', fontsize=11, fontweight='bold')
axes[1].axis('off')
colors = ['#e74c3c', '#f39c12', '#2ecc71']
labels = ['Weak\nModel 1', 'Weak\nModel 2', 'Weak\nModel 3']
for i, (color, label) in enumerate(zip(colors, labels)):
    x = 1 + i * 3.5
    axes[1].add_patch(plt.Rectangle((x-0.7, 2.5), 1.4, 1, fill=True, color=color, alpha=0.8))
    axes[1].text(x, 3, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    if i < 2:
        axes[1].annotate('', xy=(x+2.1, 3), xytext=(x+0.7, 3),
                         arrowprops=dict(arrowstyle='->', color='gray', lw=2))
        axes[1].text(x+1.5, 3.3, 'Fix\nErrors', ha='center', fontsize=7, color='purple')
axes[1].add_patch(plt.Rectangle((3.5, 0.5), 3, 1, fill=True, color='#9b59b6', alpha=0.8))
axes[1].text(5, 1, 'Weighted Sum', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.suptitle('Ensemble Methods: Bagging vs Boosting', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2️⃣ Boosting Methods Overview

### 🔑 Boosting Methods Ka Family

| Method | Year | Key Idea | Best For |
|--------|------|----------|----------|
| **AdaBoost** | 1997 | Sample weights update karo | Binary classification, simple |
| **Gradient Boosting (GBM)** | 1999 | Residuals (errors) pe next model train karo | Classification + Regression |
| **XGBoost** | 2014 | GBM + regularization + speed | Competitions, large datasets |
| **LightGBM** | 2017 | GBM + leaf-wise growth (faster) | Very large datasets |
| **CatBoost** | 2017 | GBM + categorical features auto-handle | Mixed type data |

**Aaj hum sikhenge:** AdaBoost aur Gradient Boosting (GBM) — yeh fundamentals hain.

---
### 🎯 Yeh Kyun Zaroori Hai?
> XGBoost aur LightGBM Gradient Boosting ke hi improved versions hain. Agar tum GBM samajh lo, toh XGBoost automatically samajh aayega. AdaBoost samajhne se boosting ka pura intuition clear hota hai.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

X = df.drop('Default', axis=1)
y = df['Default']

# Comparison of methods on sample data
models = {
    'Decision Tree':     DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost':          AdaBoostClassifier(n_estimators=50, random_state=42, algorithm='SAMME'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, random_state=42)
}

print(f"{'Model':<25} {'CV Mean':>10} {'CV Std':>10}")
print("-" * 50)
for name, model in models.items():
    cv = cross_val_score(model, X, y, cv=3)  # cv=3 small dataset ke liye
    print(f"{name:<25} {cv.mean():>10.4f} {cv.std():>10.4f}")

print("\n📌 Note: Small dataset pe results vary kar sakte hain.")
print("Zyada data pe Boosting methods consistently better perform karte hain.")

---
## 3️⃣ AdaBoost — Theory

### 🔑 Naye Keywords

| Term | Matlab |
|------|--------|
| **Sample Weight (w_i)** | Har training sample ko kitna importance do |
| **Error Rate (ε)** | Is iteration mein weak learner ne kitne % galat kiya |
| **Alpha (α)** | Is weak learner ko final vote mein kitna weight milega |
| **Decision Stump** | depth=1 ka decision tree — AdaBoost ka default weak learner |
| **Weighted Majority Vote** | Final prediction = sab weak learners ka weighted sum |

### How AdaBoost Works — Full Algorithm

**Step 1: Sab samples ko equal weight do**
$$w_i = \frac{1}{N} \quad \text{(N = total samples)}$$

**Step 2: Weak learner fit karo (iteration t)**
- Weighted Gini use karta hai splits ke liye

**Step 3: Error rate calculate karo**
$$\varepsilon_t = \frac{\sum_{i: y_i \neq h_t(x_i)} w_i}{\sum_i w_i}$$

**Step 4: Learner weight (alpha) calculate karo**
$$\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \varepsilon_t}{\varepsilon_t}\right)$$

- ε = 0.1 → α = 1.1 (zyada accurate → zyada vote)
- ε = 0.5 → α = 0 (random guessing → koi vote nahi)

**Step 5: Sample weights update karo**
- Galat predict: $w_i \leftarrow w_i \cdot e^{+\alpha_t}$ (weight BADHAO)
- Sahi predict: $w_i \leftarrow w_i \cdot e^{-\alpha_t}$ (weight GHATAO)
- Phir normalize karo: $w_i \leftarrow \frac{w_i}{\sum_j w_j}$

**Step 6: T iterations ke baad Final Prediction**
$$H(x) = \text{sign}\left(\sum_{t=1}^{T} \alpha_t \cdot h_t(x)\right)$$

### Visual Intuition
```
Round 1: Stump 1 fit karo (sab weights = 1/15)
         → 3 samples galat predict hue
         → ε₁ = 0.2, α₁ = 0.69
         → Galat samples ke weights 2x ho gaye!

Round 2: Stump 2 fit karo (high weight wale samples pe)
         → Different 2 samples galat predict hue
         → ε₂ = 0.15, α₂ = 0.85

Round 3: Stump 3...
...
Final: H(x) = sign(α₁·h₁ + α₂·h₂ + α₃·h₃ + ...)
```

---
### 🎯 Yeh Kyun Zaroori Hai?
> AdaBoost ka core idea: **galat samples pe zyada dhyan do**. Yeh exactly wahi hai jo ek achha teacher karta hai — jo concepts miss ho rahe hain unpe zyada practice karo. Sample weights ka concept baad mein XGBoost ke regularization mein bhi kaam aata hai.

In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# AdaBoost - Manual Step-by-Step
# ==========================================

# Simple 2-feature dataset
X_simple = df[['Credit_Score', 'Debt_Ratio']].values
y_simple  = np.where(df['Default'] == 1, -1, 1)  # AdaBoost: -1 / +1
N = len(y_simple)

# Step 1: Equal weights
weights = np.ones(N) / N
print(f"Step 1: Initial Weights (all = {1/N:.4f})")
print(f"  {weights.round(4)}")

from sklearn.tree import DecisionTreeClassifier

print("\n" + "="*60)
print("AdaBoost Iterations:")
print("="*60)

alphas = []
stumps = []

for t in range(1, 4):  # 3 iterations
    # Step 2: Weak learner (stump) fit karo
    stump = DecisionTreeClassifier(max_depth=1, random_state=t)
    stump.fit(X_simple, y_simple, sample_weight=weights)
    pred = stump.predict(X_simple)

    # Step 3: Weighted error
    incorrect = (pred != y_simple)
    epsilon = np.sum(weights[incorrect]) / np.sum(weights)
    epsilon = np.clip(epsilon, 1e-10, 1 - 1e-10)  # avoid div by zero

    # Step 4: Alpha
    alpha = 0.5 * np.log((1 - epsilon) / epsilon)

    # Step 5: Update weights
    weights *= np.exp(-alpha * y_simple * pred)
    weights /= np.sum(weights)  # normalize

    alphas.append(alpha)
    stumps.append(stump)

    print(f"\nRound {t}:")
    print(f"  Galat predictions: {incorrect.sum()} samples")
    print(f"  Error rate (ε):    {epsilon:.4f}")
    print(f"  Learner weight (α):{alpha:.4f}")
    print(f"  New weights (top 5): {weights[:5].round(4)}")

print(f"\n📌 Alpha values: {[round(a, 3) for a in alphas]}")
print("Zyada accurate stump ko zyada alpha (vote weight) milta hai!")

In [ ]:
# Alpha vs Error Rate relationship visualize karo
eps_range = np.linspace(0.01, 0.49, 100)
alpha_range = 0.5 * np.log((1 - eps_range) / eps_range)

plt.figure(figsize=(8, 4))
plt.plot(eps_range, alpha_range, 'b-', linewidth=2.5)
plt.axvline(0.5, color='red', linestyle='--', label='Random Guessing (ε=0.5, α=0)')
plt.scatter([0.1, 0.2, 0.3, 0.4], 
            [0.5*np.log(0.9/0.1), 0.5*np.log(0.8/0.2), 0.5*np.log(0.7/0.3), 0.5*np.log(0.6/0.4)],
            color='red', s=80, zorder=5)
plt.xlabel('Error Rate (ε)', fontsize=11)
plt.ylabel('Alpha (α) — Learner Weight', fontsize=11)
plt.title('AdaBoost: Error Rate vs Learner Weight (Alpha)', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📌 Key Insight:")
print("  Low error (ε=0.1)  → High alpha (zyada vote)")
print("  High error (ε=0.4) → Low alpha (kam vote)")
print("  ε=0.5 (random)     → Alpha=0 (koi vote nahi)")

---
## 4️⃣ AdaBoost — sklearn Implementation

### 🔑 Naye Keywords

| Parameter | Matlab |
|-----------|--------|
| `n_estimators` | Kitne weak learners train karne hain |
| `learning_rate` | Har estimator ka contribution kitna scale karo |
| `base_estimator` | Weak learner kaunsa use karo (default: Decision Stump) |
| `algorithm` | 'SAMME' (discrete) ya 'SAMME.R' (real, probabilistic) |

### Old vs New Method

| Task | Old Way | New Way |
|------|---------|--------|
| Confusion Matrix plot | `confusion_matrix()` + manual `imshow()` | `ConfusionMatrixDisplay.from_estimator()` |
| Metrics | Print karo manually | `classification_report()` |

---
### 🎯 Kyun Zaroori Hai?
> `ConfusionMatrixDisplay` ek line mein confusion matrix plot karta hai — koi extra matplotlib code nahi chahiye. Industry mein yahi use hota hai.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay  # NEW METHOD
)
from sklearn.metrics import confusion_matrix  # OLD METHOD (comparison ke liye)
import matplotlib.pyplot as plt

X = df.drop('Default', axis=1)
y = df['Default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ===========================================
# AdaBoost Model
# ===========================================
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision Stump
    n_estimators=100,
    learning_rate=1.0,
    algorithm='SAMME',
    random_state=42
)
ada.fit(X_train, y_train)

y_pred = ada.predict(X_test)
print(f"Train Accuracy: {ada.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
cv = cross_val_score(ada, X, y, cv=3)
print(f"CV Score:       {cv.mean():.4f} ± {cv.std():.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# ==================================================
# Confusion Matrix — OLD WAY vs NEW WAY
# ==================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- OLD WAY ---
cm = confusion_matrix(y_test, y_pred)
axes[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()/2 else 'black')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Predicted 0', 'Predicted 1'])
axes[0].set_yticklabels(['Actual 0', 'Actual 1'])
axes[0].set_title('OLD Way: confusion_matrix() + imshow()\n(More code, manual work)', fontsize=10)

# --- NEW WAY ---
ConfusionMatrixDisplay.from_estimator(
    ada, X_test, y_test,
    display_labels=['No Default', 'Default'],
    cmap='Blues',
    ax=axes[1]
)
axes[1].set_title('NEW Way: ConfusionMatrixDisplay.from_estimator()\n(1 line, clean, industry standard!)', fontsize=10)

plt.suptitle('Confusion Matrix: Old vs New Method', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📌 Confusion Matrix Terminology:")
print("  TN (True Negative):  Sahi predict kiya — Default nahi hoga (aur hua bhi nahi)")
print("  TP (True Positive):  Sahi predict kiya — Default hoga (aur hua bhi)")
print("  FP (False Positive): Galat — Default hoga predict kiya, par nahi hua")
print("  FN (False Negative): Galat — Default nahi hoga predict kiya, par hua! ← Dangerous!")

In [ ]:
# n_estimators ka effect
n_range = range(10, 201, 10)
train_scores, test_scores = [], []

for n in n_range:
    model = AdaBoostClassifier(n_estimators=n, algorithm='SAMME', random_state=42)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

plt.figure(figsize=(9, 4))
plt.plot(n_range, train_scores, 'b-', label='Train')
plt.plot(n_range, test_scores, 'r-', label='Test')
plt.xlabel('n_estimators')
plt.ylabel('Accuracy')
plt.title('AdaBoost: n_estimators vs Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("📌 AdaBoost typically does NOT overfit as n_estimators badhta hai — unlike single trees!")

---
## 5️⃣ Gradient Boosting — Theory

### 🔑 Naye Keywords

| Term | Matlab |
|------|--------|
| **Residual** | Actual value − Predicted value = Error |
| **Pseudo-Residual** | Gradient descent ka direction (loss function ki derivative) |
| **Loss Function** | Model ka total error measure |
| **Learning Rate (ν)** | Har tree ka contribution kitna scale karo (shrinkage) |
| **Shrinkage** | Learning rate se multiply karna — overfitting rokta hai |

### Gradient Boosting vs AdaBoost

| | AdaBoost | Gradient Boosting |
|-|----------|------------------|
| **Focus** | Sample weights | Residuals (errors) |
| **Weak learner** | Decision Stump only | Any tree (depth=3-5 common) |
| **Learning** | Sample reweighting | Gradient descent in function space |
| **More flexible?** | No | Yes |

### How Gradient Boosting Works

**Step 1:** Initial prediction = mean of target
$$F_0(x) = \bar{y}$$

**Step 2:** Residuals calculate karo
$$r_i = y_i - F_0(x_i)$$

**Step 3:** Residuals pe ek tree fit karo

**Step 4:** Model update karo
$$F_1(x) = F_0(x) + \nu \cdot h_1(x)$$

**Step 5:** Naye residuals pe phir repeat karo...

$$F_M(x) = F_0(x) + \nu \sum_{m=1}^{M} h_m(x)$$

### Real Numbers Ke Saath Example

```
Target: [1, 0, 1, 0, 1] (Credit Default)

Round 0: Initial prediction = mean = 0.6
  Residuals = [1-0.6, 0-0.6, 1-0.6, 0-0.6, 1-0.6]
            = [+0.4,  -0.6,  +0.4,  -0.6,  +0.4]

Round 1: Tree fits residuals!
  → Learns: high credit score → positive residual (default)
  → Update: F1 = 0.6 + 0.1 × h1(x)

Round 2: Naye residuals pe tree fit...
  → Bacha hua error aur kam hota jaata hai!
```

---
### 🎯 Yeh Kyun Zaroori Hai?
> Gradient Boosting ek **mathematical framework** hai. Jab aap XGBoost ka documentation padhoge, wahan bhi same concepts milenge: loss function, gradient, residuals. Yeh concept samajhne ke baad aap koi bhi boosting library use kar sakte ho.

In [ ]:
import numpy as np

# ==========================================
# Gradient Boosting — Residual Concept Demo
# ==========================================
# Regression example se samjhte hain (simple numbers)

y_actual   = np.array([1, 0, 1, 0, 1, 1, 0, 1, 0, 0], dtype=float)
F0         = np.full(10, y_actual.mean())  # Initial: mean predict karo
lr         = 0.5  # learning rate

print("=" * 55)
print(f"Actual values: {y_actual}")
print(f"F0 (initial pred = mean = {y_actual.mean():.1f}): {F0}")

predictions = [F0.copy()]
mse_per_round = [np.mean((y_actual - F0)**2)]

from sklearn.tree import DecisionTreeRegressor
X_demo = np.arange(10).reshape(-1, 1)  # Simple feature

F = F0.copy()
for round_num in range(1, 4):
    residuals = y_actual - F

    # Tree to fit residuals
    tree = DecisionTreeRegressor(max_depth=1, random_state=round_num)
    tree.fit(X_demo, residuals)
    h = tree.predict(X_demo)

    F = F + lr * h
    mse = np.mean((y_actual - F)**2)
    predictions.append(F.copy())
    mse_per_round.append(mse)

    print(f"\nRound {round_num}:")
    print(f"  Residuals:   {residuals.round(3)}")
    print(f"  h(x):        {h.round(3)}")
    print(f"  F{round_num} (updated): {F.round(3)}")
    print(f"  MSE:         {mse:.4f}")

print("\n📌 MSE per round:", [round(m, 4) for m in mse_per_round])
print("Error rounds ke saath GHATTA JAA raha hai — Gradient Boosting kaam kar raha hai!")

In [ ]:
# Residuals ka visualization
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (pred, title) in enumerate(zip(
    [predictions[0], predictions[1], predictions[3]],
    ['Round 0 (Initial)', 'Round 1', 'Round 3']
)):
    residuals_plot = y_actual - pred
    colors = ['#e74c3c' if r > 0 else '#3498db' for r in residuals_plot]
    axes[i].bar(range(10), residuals_plot, color=colors, edgecolor='black', alpha=0.8)
    axes[i].axhline(0, color='black', linewidth=1.5)
    axes[i].set_title(f'{title}\nMSE={np.mean(residuals_plot**2):.4f}')
    axes[i].set_xlabel('Sample Index')
    axes[i].set_ylabel('Residual')
    axes[i].set_ylim(-0.8, 0.8)

plt.suptitle('Gradient Boosting: Residuals Shrink with Each Round', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("📌 Residuals (bars) round ke saath chote hote ja rahe hain — model improve ho raha hai!")

---
## 6️⃣ Gradient Boosting — sklearn Implementation

### 🔑 Key Parameters

| Parameter | Matlab | Typical Values |
|-----------|--------|---------------|
| `n_estimators` | Kitne trees fit karne hain | 100-500 |
| `learning_rate` | Har tree ki contribution | 0.01-0.3 |
| `max_depth` | Har tree ki depth | 3-5 |
| `subsample` | Har tree ke liye % data use karo | 0.8-1.0 |
| `min_samples_split` | Split ke liye min samples | 2-20 |

### Learning Rate vs n_estimators Tradeoff
```
High learning_rate + Few estimators = Fast but rough
Low learning_rate  + Many estimators = Slower but better accuracy

Rule: learning_rate × n_estimators ≈ constant
```

---
### 🎯 Real World Mein Kahan Use Hoga?
> **Finance:** Stock price movement predict karo | **Healthcare:** Patient readmission predict karo | **E-commerce:** Customer lifetime value predict karo | **HR:** Employee attrition predict karo

> Gradient Boosting tabular data pe almost always Random Forest se better hai. Industry mein GBM ka optimized version XGBoost ya LightGBM use karte hain — same concept, faster.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

X = df.drop('Default', axis=1)
y = df['Default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# ===========================================
# Gradient Boosting Model
# ===========================================
gbm = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
gbm.fit(X_train, y_train)

y_pred = gbm.predict(X_test)
print(f"Train Accuracy: {gbm.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {gbm.score(X_test, y_test):.4f}")
cv = cross_val_score(gbm, X, y, cv=3)
print(f"CV Score:       {cv.mean():.4f} ± {cv.std():.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

In [ ]:
# Feature Importance
importances = pd.Series(gbm.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Feature Importance
importances.plot(kind='barh', ax=axes[0],
                 color=['#e74c3c' if v > 0.2 else '#3498db' for v in importances],
                 edgecolor='black', alpha=0.85)
axes[0].set_title('GBM Feature Importance')
axes[0].set_xlabel('Importance Score')

# Confusion Matrix (NEW WAY)
ConfusionMatrixDisplay.from_estimator(
    gbm, X_test, y_test,
    display_labels=['No Default', 'Default'],
    cmap='Greens',
    ax=axes[1]
)
axes[1].set_title('GBM Confusion Matrix')

plt.suptitle('Gradient Boosting — Results', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Learning Rate aur n_estimators tradeoff
lr_values = [0.01, 0.05, 0.1, 0.3, 1.0]
n_estimators_range = [50, 100, 200]

results = []
for lr in lr_values:
    for n in n_estimators_range:
        model = GradientBoostingClassifier(n_estimators=n, learning_rate=lr,
                                           max_depth=3, random_state=42)
        model.fit(X_train, y_train)
        cv = cross_val_score(model, X, y, cv=3).mean()
        results.append({'lr': lr, 'n': n, 'CV': round(cv, 4)})

pivot = pd.DataFrame(results).pivot(index='lr', columns='n', values='CV')
print("Learning Rate vs n_estimators — CV Accuracy:")
print(pivot.to_string())
print("\n📌 Low lr + High n_estimators = usually best (but slower to train)")

In [ ]:
# ==================================================
# FINAL COMPARISON: AdaBoost vs GBM vs Decision Tree
# ==================================================
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

models_final = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost':      AdaBoostClassifier(n_estimators=100, algorithm='SAMME', random_state=42),
    'Gradient Boost':GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

summary = []
for name, model in models_final.items():
    model.fit(X_train, y_train)
    cv = cross_val_score(model, X, y, cv=3)
    summary.append({
        'Model': name,
        'Train Acc': round(model.score(X_train, y_train), 4),
        'Test Acc':  round(model.score(X_test, y_test), 4),
        'CV Mean':   round(cv.mean(), 4),
        'CV Std':    round(cv.std(), 4)
    })

print(pd.DataFrame(summary).to_string(index=False))
print("\n✅ Boosting methods generally give better CV scores on complex datasets!")

---
## 🎯 Poore Notebook Ka Summary — Humne Kya Seekha?

| Concept | Key Point | Real World Use |
|---------|-----------|---------------|
| Boosting | Weak learners ko sequentially combine karo | Tabular data pe best accuracy |
| AdaBoost | Sample weights update karo — galat pe focus | Fraud detection, email spam |
| Alpha (α) | Accurate learner ko zyada vote milta hai | Model interpretability |
| Gradient Boosting | Residuals pe next model train karo | Finance, healthcare, HR |
| Learning Rate | Shrinkage — overfitting control | Hyperparameter tuning |
| ConfusionMatrixDisplay | 1 line mein matrix plot | Industry standard reporting |

### Aage Kya Seekhein?
```
AdaBoost → Gradient Boosting → XGBoost → LightGBM → CatBoost
   (Intro)      (Core theory)   (Faster+better)  (Production)
```

**Next Step:** Notebook 02 mein yeh sab concepts ek real Kaggle dataset pe apply karenge!